In [ ]:
!pip install -q tensorflow

In [ ]:
import json, os, random, zipfile, math
import numpy as np
import tensorflow as tf
from tensorflow import keras

print("TensorFlow version:", tf.__version__)

## 1. Generate Synthetic KSA-Lifestyle Reminder History (with Titles)

In [ ]:
CATEGORIES = ["personal", "work", "family", "other"]
CAT_IDX    = {c: i for i, c in enumerate(CATEGORIES)}

random.seed(42)
np.random.seed(42)

# ── Title word pools per category ────────────────────────────────────────────
TITLE_POOLS = {
    "personal": [
        "prayer", "fajr", "dhuhr", "asr", "maghrib", "isha",
        "gym", "exercise", "medication", "vitamins", "quran",
        "meditation", "skincare", "journal",
    ],
    "work": [
        "meeting", "deadline", "report", "email", "presentation",
        "review", "project", "task", "call", "submit",
        "lecture", "homework", "study", "exam",
    ],
    "family": [
        "dinner", "lunch", "kids", "school", "doctor",
        "hospital", "family", "cook", "visit", "gather",
        "birthday", "picnic", "appointment", "collect",
    ],
    "other": [
        "groceries", "shopping", "buy", "errand", "store",
        "pharmacy", "pay", "bank", "car", "laundry",
        "clean", "fix", "return", "pickup",
    ],
}

records = []

def add_records(n, day_range, hour_mean, hour_std, category):
    pool = TITLE_POOLS[category]
    for _ in range(n):
        day  = random.choice(list(day_range))
        hour = float(np.clip(np.random.normal(hour_mean, hour_std), 0, 23.99))
        # Pick 1–2 random title words from the category pool
        num_words = random.choice([1, 2])
        title = " ".join(random.sample(pool, min(num_words, len(pool))))
        records.append((day, round(hour, 2), CAT_IDX[category], title))

# ── KSA Daily Patterns (work week: Sun–Thu = 0–4; weekend: Fri–Sat = 5–6) ──

# Prayer-related (personal)
add_records(80,  range(7),    4.75,  0.4, "personal")   # Fajr    ~4:45
add_records(60,  range(7),   12.25,  0.3, "personal")   # Dhuhr  ~12:15
add_records(50,  range(7),   15.5,   0.3, "personal")   # Asr     ~3:30
add_records(50,  range(7),   18.25,  0.3, "personal")   # Maghrib ~6:15
add_records(50,  range(7),   20.0,   0.3, "personal")   # Isha    ~8:00

# Work / school (Sun–Thu)
add_records(100, range(5),    7.5,   0.5, "work")       # Morning commute
add_records(80,  range(5),   10.0,   0.8, "work")       # Mid-morning tasks
add_records(60,  range(5),   14.0,   0.6, "work")       # Afternoon work

# Family
add_records(70,  range(7),   13.0,   0.7, "family")     # Lunch
add_records(80,  range(7),   19.0,   0.8, "family")     # Dinner / family time
add_records(60,  [5, 6],     11.0,   1.5, "family")     # Weekend outings

# Other / errands
add_records(60,  range(5),   17.0,   1.0, "other")      # After-work errands
add_records(70,  [5, 6],     21.0,   1.0, "other")      # Weekend evening shopping
add_records(40,  range(7),   22.0,   0.8, "other")      # Late-night café / social

random.shuffle(records)
print(f"Generated {len(records)} synthetic records")

from collections import Counter
dist = Counter(CATEGORIES[c] for _, _, c, _ in records)
for cat, count in dist.most_common():
    print(f"  {cat:10s}: {count}")

In [ ]:

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

DAY_NAMES  = ["Sun", "Mon", "Tue", "Wed", "Thu", "Fri", "Sat"]
CAT_COLORS = {"personal": "#4C9BE8", "work": "#E8834C", "family": "#4CE87A", "other": "#C84CE8"}

# ── 1. Sample table: first 20 records ────────────────────────────────────────
df = pd.DataFrame(records, columns=["day_idx", "hour", "cat_idx", "title"])
df["day"]      = df["day_idx"].map(lambda d: DAY_NAMES[d])
df["category"] = df["cat_idx"].map(lambda c: CATEGORIES[c])
df["time"]     = df["hour"].map(lambda h: f"{int(h):02d}:{int((h % 1)*60):02d}")

print("── Sample of 20 generated records (with titles) ──")
print(df[["day", "time", "category", "title"]].head(20).to_string(index=False))

# ── 2. Scatter plot: day vs hour, colored by category ────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))
for cat, color in CAT_COLORS.items():
    subset = df[df["category"] == cat]
    ax.scatter(subset["day_idx"] + (np.random.rand(len(subset)) - 0.5) * 0.4,
               subset["hour"],
               c=color, alpha=0.5, s=20, label=cat)

ax.set_xticks(range(7))
ax.set_xticklabels(DAY_NAMES)
ax.set_ylabel("Hour of day")
ax.set_xlabel("Day of week")
ax.set_title(f"Generated dataset — {len(records)} samples")
ax.set_ylim(0, 24)
ax.set_yticks(range(0, 25, 2))
ax.grid(axis='y', alpha=0.3)
ax.legend(title="Category", loc="upper right")
plt.tight_layout()
plt.show()

# ── 3. Category counts bar chart ─────────────────────────────────────────────
counts = df["category"].value_counts().reindex(CATEGORIES)
fig2, ax2 = plt.subplots(figsize=(6, 3))
ax2.bar(counts.index, counts.values,
        color=[CAT_COLORS[c] for c in counts.index])
for i, (cat, val) in enumerate(zip(counts.index, counts.values)):
    ax2.text(i, val + 3, str(val), ha='center', fontsize=10)
ax2.set_ylabel("# samples")
ax2.set_title("Samples per category")
ax2.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()


## 2. Build Vocabulary & Encode Features

We build a **fixed vocabulary** of keywords from all title pools, then encode each record as:
- 4 cyclic time features (day sin/cos, hour sin/cos)
- 30 keyword flags (1 if word is in the title, 0 otherwise)

Total input: **34 features** — inspired by the MS-LaTTE paper's lexical baseline approach.

In [ ]:
# ── Build a fixed vocabulary from all title pools ────────────────────────────
all_words = set()
for pool in TITLE_POOLS.values():
    all_words.update(pool)

VOCAB = sorted(all_words)  # Fixed order — saved to vocabulary.json
VOCAB_IDX = {w: i for i, w in enumerate(VOCAB)}

print(f"Vocabulary size: {len(VOCAB)}")
print(f"Words: {VOCAB}")

def encode_title(title):
    """Encode a title string as a binary vector over the vocabulary."""
    words = title.lower().split()
    vec = [0.0] * len(VOCAB)
    for w in words:
        if w in VOCAB_IDX:
            vec[VOCAB_IDX[w]] = 1.0
    return vec

def encode_features(day, hour, title=""):
    """
    Encode day-of-week (0=Sun … 6=Sat), hour (0–24), and title text
    as a feature vector: 4 cyclic + len(VOCAB) keyword flags.
    """
    day_sin  = math.sin(2 * math.pi * day  / 7)
    day_cos  = math.cos(2 * math.pi * day  / 7)
    hour_sin = math.sin(2 * math.pi * hour / 24)
    hour_cos = math.cos(2 * math.pi * hour / 24)
    return [day_sin, day_cos, hour_sin, hour_cos] + encode_title(title)

# ── Encode all records ───────────────────────────────────────────────────────
X = np.array([encode_features(d, h, t) for d, h, _, t in records], dtype=np.float32)
Y = np.array([c for _, _, c, _ in records], dtype=np.int32)
Y_onehot = np.eye(len(CATEGORIES), dtype=np.float32)[Y]

split = int(len(X) * 0.8)
X_train, X_test = X[:split], X[split:]
Y_train, Y_test = Y_onehot[:split], Y_onehot[split:]
Y_test_idx = Y[split:]

print(f"\nTrain: {len(X_train)}  |  Test: {len(X_test)}")
print(f"Input shape: {X_train.shape}  |  Output shape: {Y_train.shape}")

## 3. Build & Train Model

In [ ]:
INPUT_DIM = 4 + len(VOCAB)  # 4 temporal + keyword flags

model = keras.Sequential([
    keras.layers.Dense(32, activation='relu', input_shape=(INPUT_DIM,)),
    keras.layers.Dropout(0.2),
    keras.layers.Dense(16, activation='relu'),
    keras.layers.Dense(len(CATEGORIES), activation='softmax'),
])

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy'],
)

model.summary()

history = model.fit(
    X_train, Y_train,
    validation_split=0.15,
    epochs=50,
    batch_size=32,
    verbose=1,
)

## 4. Evaluate & Sample Predictions

In [ ]:
preds       = model.predict(X_test)
pred_labels = np.argmax(preds, axis=-1)
accuracy    = (pred_labels == Y_test_idx).mean()
print(f"Test accuracy: {accuracy:.2%}")

# ── Sample predictions: time + title combined ────────────────────────────────
test_scenarios = [
    (0,  5.0,  "prayer fajr",      "Sunday    5:00 AM  — Fajr prayer"),
    (0,  8.0,  "meeting",          "Sunday    8:00 AM  — Work meeting"),
    (1, 13.0,  "lunch family",     "Monday    1:00 PM  — Family lunch"),
    (2, 17.0,  "groceries buy",    "Tuesday   5:00 PM  — Buy groceries"),
    (3, 19.0,  "dinner kids",      "Wednesday 7:00 PM  — Dinner with kids"),
    (5, 11.0,  "picnic family",    "Friday   11:00 AM  — Family picnic"),
    (5, 21.0,  "shopping",         "Friday    9:00 PM  — Shopping"),
    (6, 22.0,  "pharmacy",         "Saturday 10:00 PM  — Pharmacy run"),
    # ── Text-only tests (does text override wrong time?) ──
    (0, 21.0,  "meeting deadline", "Sunday    9:00 PM  — Work (wrong time)"),
    (2,  5.0,  "groceries store",  "Tuesday   5:00 AM  — Errands (wrong time)"),
    # ── No text (falls back to time only) ──
    (0,  5.0,  "",                 "Sunday    5:00 AM  — No title"),
    (3, 19.0,  "",                 "Wednesday 7:00 PM  — No title"),
]

print("\n── Sample Predictions ──")
for day, hour, title, desc in test_scenarios:
    x    = np.array([encode_features(day, hour, title)], dtype=np.float32)
    p    = model.predict(x, verbose=0)[0]
    cat  = CATEGORIES[np.argmax(p)]
    conf = np.max(p)
    print(f"  {desc:42s} → {cat:10s} ({conf:.0%})")

## 5. Export to TFLite & Download ZIP

In [ ]:
converter    = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

OUT_DIR = "habit_output"
os.makedirs(OUT_DIR, exist_ok=True)

with open(f"{OUT_DIR}/model.tflite", "wb") as f:
    f.write(tflite_model)

with open(f"{OUT_DIR}/categories.json", "w") as f:
    json.dump(CATEGORIES, f)

# Save the vocabulary — Flutter needs this to encode titles the same way
with open(f"{OUT_DIR}/vocabulary.json", "w") as f:
    json.dump(VOCAB, f)

config = {
    "input_features": ["day_sin", "day_cos", "hour_sin", "hour_cos"] + [f"kw_{w}" for w in VOCAB],
    "num_categories": len(CATEGORIES),
    "categories": CATEGORIES,
    "vocab_size": len(VOCAB),
}
with open(f"{OUT_DIR}/config.json", "w") as f:
    json.dump(config, f, indent=2)

with zipfile.ZipFile("habit_model.zip", "w") as zf:
    for fname in os.listdir(OUT_DIR):
        zf.write(f"{OUT_DIR}/{fname}", fname)

print("✅ Done! Files inside habit_model.zip:")
print("   model.tflite  categories.json  vocabulary.json  config.json")
print(f"\n   Input dimension: {INPUT_DIM} (4 temporal + {len(VOCAB)} keywords)")
print("\nExtract into: assets/models/habit/  in your Flutter project.")

# Auto-download in Colab
try:
    from google.colab import files
    files.download("habit_model.zip")
except ImportError:
    print("(Not in Colab — find habit_model.zip in the file browser on the left)")